In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("/content/Hospital_data_completed - Hospital_data_completed.csv.csv")
df.head()


,Patient_ID,Age,Gender,Blood_Type,Chronic_Condition,BMI,Annual_Visits,Avg_Stay_Duration,Total_Spending,Insurance_Type
0,P10001,45,Male,A+,0,20.0,3,2,750.0,NaN
1,P10002,60,Female,B+,1,25.0,6,5,1550.0,Private
2,P10003,30,Male,O+,3,16.0,1,1,410.0,NaN
3,P10004,45,Female,O-,0,50.0,4,3,1300.0,NaN
4,P10005,60,Male,A-,1,30.0,8,7,2100.0,Private


In [ ]:
df.columns


Index(['Patient_ID', 'Age', 'Gender', 'Blood_Type', 'Chronic_Condition', 'BMI',
       'Annual_Visits', 'Avg_Stay_Duration', 'Total_Spending',
       'Insurance_Type'],
      dtype='object')

In [ ]:
columns_to_drop = [
    'Avg_Stay_Duration',
    'Total_Spending',
    'Insurance_Type','Annual_Visits','BMI'
]

df = df.drop(columns=columns_to_drop, errors='ignore')


In [ ]:
print("Columns after dropping unwanted ones:")
df.columns



Columns after dropping unwanted ones:


Index(['Patient_ID', 'Age', 'Gender', 'Blood_Type', 'Chronic_Condition'], dtype='object')

In [ ]:
df = df.rename(columns={
    'Chronic_Condition': 'agglutination grading'
})


In [ ]:
print("Final column names:")
df.columns


Final column names:


Index(['Patient_ID', 'Age', 'Gender', 'Blood_Type', 'agglutination grading'], dtype='object')

In [ ]:
df.head()


,Patient_ID,Age,Gender,Blood_Type,agglutination grading
0,P10001,45,Male,A+,0
1,P10002,60,Female,B+,1
2,P10003,30,Male,O+,3
3,P10004,45,Female,O-,0
4,P10005,60,Male,A-,1


In [ ]:
df.shape


(129, 5)

In [ ]:
df["Blood_Type"] = df["Blood_Type"].astype(str).str.strip().str.upper()

In [ ]:
# Blood compatibility rules (RBC transfusion logic)
donation_map = {
    "O-": ["O-", "O+", "A-", "A+", "B-", "B+", "AB-", "AB+"],
"O+": ["O+", "A+", "B+", "AB+"],
"A-": ["A-", "A+", "AB-", "AB+"],
"A+": ["A+", "AB+"],
"B-": ["B-", "B+", "AB-", "AB+"],
"B+": ["B+", "AB+"],
"AB-": ["AB-", "AB+"],
"AB+": ["AB+"]
}

# Reverse logic: who can donate to a given blood type
def get_acceptable_donors(recipient_blood):
    donors = []
    for donor, recipients in donation_map.items():
        if recipient_blood in recipients:
            donors.append(donor)
    return donors


In [ ]:
################
blood_input = input("Enter Blood Group (e.g., O+, A-, AB+): ").strip().upper()
grade_input = int(input("Enter Agglutination Grade (0–4): "))


Enter Blood Group (e.g., O+, A-, AB+): AB+
Enter Agglutination Grade (0–4): 4


In [ ]:
# Extract ABO and Rh separately df["ABO_Type"] = df["Blood_Type"].str[:-1] df["Rh_Factor"] = df["Blood_Type"].str[-1] # Get donor list compatible_donors = get_acceptable_donors(blood_input) print("Compatible donors:", compatible_donors) # Filter donors_df = df[ (df["Blood_Type"].isin(compatible_donors)) & (df["agglutination grading"] == grade_input) ]

In [ ]:
compatible_donors = get_acceptable_donors(blood_input)

In [ ]:
# Basic filtering first (ABO + Grade)
donors_df = df[
    (df["Blood_Type"].isin(compatible_donors)) &
    (df["agglutination grading"] == grade_input)
]

# Extract recipient Rh
recipient_rh = blood_input[-1]

# Apply Rh restriction
if recipient_rh == "-":
    donors_df = donors_df[donors_df["Rh_Factor"] == "-"]

# Select display columns
donors_df = donors_df[["Patient_ID", "Age", "Gender", "Blood_Type", "agglutination grading"]]


In [ ]:
print("\n🟢 Serologically Compatible Recipients (Grade-Matched)")
if recipients_df.empty:
    print("No compatible recipients found.")
else:
    display(recipients_df)

print("\n🔵 Serologically Compatible Donors (Grade-Matched)")
if donors_df.empty:
    print("No compatible donors found.")
else:
    display(donors_df)


🟢 Serologically Compatible Recipients (Grade-Matched)


,Patient_ID,Age,Gender,Blood_Type,agglutination grading
4,P10005,60,Male,A-,1
12,P10013,45,Male,A-,1
20,P10021,30,Male,A-,1
28,P10029,60,Male,A-,1
36,P10037,45,Male,A-,1
44,P10045,30,Male,A-,1
52,P10053,60,Male,A-,1
60,P10061,45,Male,A-,1
68,P10069,30,Male,A-,1
76,P10077,60,Male,A-,1



🔵 Serologically Compatible Donors (Grade-Matched)


,Patient_ID,Age,Gender,Blood_Type,agglutination grading
7,P10008,60,Female,AB-,4
15,P10016,45,Female,AB-,4
23,P10024,30,Female,AB-,4
31,P10032,60,Female,AB-,4
39,P10040,45,Female,AB-,4
47,P10048,30,Female,AB-,4
55,P10056,60,Female,AB-,4
63,P10064,45,Female,AB-,4
71,P10072,30,Female,AB-,4
79,P10080,60,Female,AB-,4


In [ ]:
# Who this person can donate to
compatible_recipients = donation_map.get(blood_input, [])

recipients_df = df[
    (df["Blood_Type"].isin(compatible_recipients)) &
    (df["agglutination grading"] == grade_input)
][["Patient_ID", "Age", "Gender", "Blood_Type", "agglutination grading"]]

In [ ]:
print(df["Blood_Type"].unique())

['A+' 'B+' 'O+' 'O-' 'A-' 'B-' 'AB+' 'AB-']


In [ ]:
print("Compatible donors:", get_acceptable_donors("B+"))

Compatible donors: ['O-', 'O+', 'B-', 'B+']
